In [ ]:
import numpy as np
import anndata as an
import scanpy as sc
import scanpy.external as sce
import scipy
import os
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

In [ ]:
adata = sc.read_h5ad('../reheatHeart/reheatHeart.h5ad')
adata

In [ ]:
adata.obs.groupby(['study', 'annotation_MOFA']).size()

In [ ]:
def cosine_similarity(X, device='cuda'):
    X = X.T
    X = torch.tensor(X).to('cuda')
    X_normalized = F.normalize(X, p=2, dim=1)
    cos = X_normalized @ X_normalized.T 
    cos= cos.cpu().numpy()
    return cos

In [ ]:
studies = ['Chaffin_2022','Kuppe_2022','Koenig_2022','Reichart_2022','Simonson_2023']
cluster_key = 'annotation_MOFA'

for study in studies:
    study_adj = []
    adata_study = adata[adata.obs['study'] == study]
    if study in ['Chaffin_2022', 'Reichart_2022']:
        for i in range(10):
            adatas = [adata_study[adata_study.obs[cluster_key]==clust] for clust in adata_study.obs[cluster_key].cat.categories]
            for dat in adatas:
                sc.pp.subsample(dat, fraction = 0.2,random_state=i)
            sample = adatas[0].concatenate(*adatas[1:])
            study_adj.append(cosine_similarity(sample.X))
        adj = np.array(study_adj).mean(axis=0)
    else:
        adj = cosine_similarity(adata_study.X)
    np.fill_diagonal(adj, 0)
    adj = np.where(adj > 0, adj, 0)
    adj = adj / adj.max()
    torch.save(adj, f'reheatHeart/data/{study}.pt')  
